# Week 8 — Day 2: QLoRA Fine-Tuning

Minimal Colab notebook.

Required:
- Model: `Qwen/Qwen2.5-1.5B-Instruct`
- 4-bit loading
- LoRA rank = 16
- Learning rate = 2e-4
- Batch size = 4
- Epochs = 3

Before running: **Runtime → Change runtime type → T4 GPU**.


## STEP 1 — Install dependencies
Run once, then **Runtime → Restart session** once. Continue from STEP 2.


In [ ]:
!pip install -q -U pyarrow datasets accelerate bitsandbytes
!pip install -q "transformers==5.16.1" "trl==1.12.0" "peft==0.20.0"
print("Install complete. Restart session once, then continue from STEP 2.")


## STEP 2 — Verify GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable T4 GPU in Colab.")
print("GPU:", torch.cuda.get_device_name(0))


## STEP 3 — Upload `train.jsonl` and `val.jsonl`


In [ ]:
from google.colab import files
files.upload()


## STEP 4 — Load dataset


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl",
        "validation": "/content/val.jsonl",
    }
)

print(dataset)
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))


## STEP 5 — Load tokenizer and format data


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    user_text = example["instruction"].strip()
    if example["input"] and example["input"].strip():
        user_text += f"\n\nInput:\n{example['input'].strip()}"

    messages = [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": example["output"].strip()},
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted_dataset = dataset.map(format_example)
print(formatted_dataset["train"][0]["text"])


## STEP 6 — Load Qwen in 4-bit


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False

print("4-bit loaded:", model.is_loaded_in_4bit)


## STEP 7 — Add LoRA


In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## STEP 8 — Create trainer


In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="/content/qwen-qlora-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    dataset_text_field="text",
    max_length=128,
    packing=False,
    optim="paged_adamw_8bit",

    # Avoids the BF16/GradScaler issue seen earlier.
    fp16=False,
    bf16=False,

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    processing_class=tokenizer,
)

print("Trainer created successfully.")


## STEP 9 — Train


In [ ]:
train_result = trainer.train()

print("Training metrics:")
print(train_result.metrics)


## STEP 10 — Evaluate


In [ ]:
eval_results = trainer.evaluate()

print("Validation metrics:")
print(eval_results)


## STEP 11 — Save trained adapter


In [ ]:
from pathlib import Path
import shutil

ADAPTER_DIR = Path("/content/final_adapter")
shutil.rmtree(ADAPTER_DIR, ignore_errors=True)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

!ls -lh /content/final_adapter


## STEP 12 — Download adapter


In [ ]:
from google.colab import files

files.download("/content/final_adapter/adapter_model.safetensors")
files.download("/content/final_adapter/adapter_config.json")


## Done

Put the downloaded files locally at:

```text
adapters/adapter_model.safetensors
adapters/adapter_config.json
```

Use the actual metrics from STEP 9 and STEP 10 in `TRAINING-REPORT.md`.
